# DM4ML Assignment 2: End-to-End Data Management Pipeline for a Recommendation System

Pipeline: Data Ingestion -> Data Validation -> Data Preparation -> Data Transformation -> Feature Management -> Model Training -> Model Evaluation -> Orchestration.

The code is organised as modules inside `recommendation-pipeline/`; the notebook creates the project, runs the pipeline and shows the results.

## 1. Project setup

In [ ]:
import os
BASE = "/kaggle/working" if os.path.exists("/kaggle/working") else os.getcwd()
os.chdir(BASE)
PROJECT = os.path.join(BASE, "recommendation-pipeline")
for d in ["configs", "src/ingestion", "src/validation", "src/preparation", "src/transformation", "src/features",
          "src/training", "src/evaluation", "dags", "scripts", "tests", "data/raw", "data/processed", "data/features",
          "data/errors", "models", "reports"]:
    os.makedirs(os.path.join(PROJECT, d), exist_ok=True)

## 2. Source code
Each cell below saves one module of the project.

In [ ]:
%%writefile recommendation-pipeline/configs/config.yaml
# Central configuration for the recommendation pipeline.
# Every stage reads from this file, so a run is fully described by (code + config + seed).
seed: 42

paths:
  raw: data/raw
  processed: data/processed
  features: data/features
  errors: data/errors
  models: models
  reports: reports

data:
  source: synthetic          # "synthetic" or "csv" (csv expects users.csv/items.csv/interactions.csv in csv_dir)
  csv_dir: data/raw
  n_users: 1500
  n_items: 600
  mean_interactions_per_user: 30
  start_date: "2023-01-01"
  end_date: "2024-12-31"
  dirty_rate: 0.025          # base rate of injected data-quality problems (synthetic only)

validation:
  reference_date: "2025-01-01"   # fixed "today" so timestamp checks are reproducible
  rating_min: 1
  rating_max: 5
  age_min: 10
  age_max: 100
  release_year_min: 1900
  max_error_rate: 0.15           # quality gate: fail the pipeline if any table loses more than this share

preparation:
  test_fraction: 0.2             # last 20% of the timeline becomes the test set
  relevance_threshold: 4.0       # rating >= 4 counts as a "relevant" item

features:
  bayes_prior_count: 10          # smoothing strength for Bayesian average rating

training:
  als:
    factors: 32
    reg: 0.1
    alpha: 20.0
    iterations: 15
  biased_mf:
    factors: 16
    reg: 0.08
    iterations: 12
    bias_reg: 5.0
  hybrid:
    tune: true
    val_fraction: 0.15           # last 15% (by time) of the training period is used for tuning
    w_content_grid: [0.0, 0.15, 0.3, 0.5, 0.75, 1.0, 1.5, 2.0]
    w_pop_grid: [0.0, 0.1]

evaluation:
  k_values: [5, 10]
  primary_k: 10

mlflow:
  tracking_uri: "sqlite:///mlflow.db"
  experiment_name: "recommendation-pipeline"


In [ ]:
%%writefile recommendation-pipeline/run_pipeline.py
"""Entry point.  python run_pipeline.py [--config configs/config.yaml] [--no-prefect]

Default: run the Prefect flow (dags/recommendation_flow.py). --no-prefect runs the same stages sequentially
in plain Python (useful if Prefect is not installed).
"""
import argparse
import os
import sys
from pathlib import Path

os.environ.setdefault("PREFECT_SERVER_ANALYTICS_ENABLED", "false")
sys.path.insert(0, str(Path(__file__).resolve().parent))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default=None)
    ap.add_argument("--no-prefect", action="store_true")
    a = ap.parse_args()
    log = Path(__file__).parent / "reports" / "pipeline.log"
    log.parent.mkdir(exist_ok=True)
    log.write_text("")
    if a.no_prefect:
        from src.pipeline import run_sequential
        run_sequential(a.config)
    else:
        from dags.recommendation_flow import recommendation_pipeline
        recommendation_pipeline(a.config)
    from scripts.make_run_summary import make
    make()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile recommendation-pipeline/recommend.py
"""Prediction CLI - top-K recommendations for a user using a trained model and the SAME FeatureStore.

    python recommend.py --user 42 --model hybrid --k 10
"""
import argparse
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp

sys.path.insert(0, str(Path(__file__).resolve().parent))
from src.features.feature_store import FeatureStore  # noqa: E402
from src.training import models  # noqa: E402,F401  (needed for unpickling)
from src.utils import load_config, read_json  # noqa: E402


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--user", type=int, required=True)
    ap.add_argument("--model", default="hybrid", choices=["popularity", "als_mf", "hybrid"])
    ap.add_argument("--k", type=int, default=10)
    a = ap.parse_args()
    cfg = load_config()
    P = cfg["_paths"]
    idx = read_json(P["processed"] / "matrix_index.json")
    uids, iids = np.array(idx["user_ids"]), np.array(idx["item_ids"])
    if a.user not in set(uids):
        sys.exit(f"user {a.user} has no training history (cold start) - fall back to the popularity model")
    u = int(np.flatnonzero(uids == a.user)[0])
    model = pickle.load(open(P["models"] / f"{a.model}.pkl", "rb"))
    M = sp.load_npz(P["processed"] / "user_item_matrix.npz").tocsr()
    s = model.score([u])[0].copy()
    s[M[u].indices] = -np.inf                                   # drop items already seen
    top = np.argsort(-s)[: a.k]
    fs = FeatureStore(cfg)
    items = pd.read_parquet(P["processed"] / "items_clean.parquet").set_index("item_id")
    uf = fs.get_user_features([a.user]).iloc[0]
    print(f"User {a.user}: avg rating {uf.user_avg_rating:.2f}, {int(uf.user_activity_count)} interactions, "
          f"preferred category = {uf.user_preferred_category}")
    pf = fs.pair_features(pd.DataFrame({"user_id": a.user, "item_id": iids[top]}))
    for rank, (j, aff) in enumerate(zip(top, pf.user_item_category_affinity), 1):
        print(f"{rank:2d}. item {iids[j]:4d} | {items.loc[iids[j], 'title']} | {items.loc[iids[j], 'category']:<28} | "
              f"score {s[j]:.3f} | category affinity {aff:.2f}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile recommendation-pipeline/src/__init__.py
# (package marker)


In [ ]:
%%writefile recommendation-pipeline/src/utils.py
"""Shared helpers: config loading, path resolution, logging, JSON IO."""
from __future__ import annotations

import json
import logging
from pathlib import Path

import yaml

ROOT = Path(__file__).resolve().parents[1]


def load_config(path: str | Path | None = None) -> dict:
    path = Path(path) if path else ROOT / "configs" / "config.yaml"
    with open(path) as f:
        cfg = yaml.safe_load(f)
    cfg["_paths"] = {}
    for key, rel in cfg["paths"].items():
        p = ROOT / rel
        p.mkdir(parents=True, exist_ok=True)
        cfg["_paths"][key] = p
    return cfg


def get_logger(name: str) -> logging.Logger:
    parent = logging.getLogger("recpipe")
    if not parent.handlers:
        parent.setLevel(logging.INFO)
        fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(name)s | %(message)s", "%H:%M:%S")
        sh = logging.StreamHandler()
        sh.setFormatter(fmt)
        parent.addHandler(sh)
        (ROOT / "reports").mkdir(exist_ok=True)
        fh = logging.FileHandler(ROOT / "reports" / "pipeline.log", mode="a")
        fh.setFormatter(fmt)
        parent.addHandler(fh)
    return logging.getLogger(f"recpipe.{name}")


def write_json(obj, path: Path) -> None:
    Path(path).write_text(json.dumps(obj, indent=2, default=str))


def read_json(path: Path):
    return json.loads(Path(path).read_text())


In [ ]:
%%writefile recommendation-pipeline/src/pipeline.py
"""Plain-Python stage runner (no Prefect needed). Shared by the Prefect flow and `--no-prefect` mode."""
from __future__ import annotations

import importlib
import json
import time
from datetime import datetime, timezone

from src.utils import ROOT, load_config

RUN_LOG = ROOT / "reports" / "run_log.jsonl"

# (task name, "module:function") in dependency order
STAGES = [
    ("ingest", "src.ingestion.ingest:run"),
    ("validate", "src.validation.validate:run"),
    ("prepare", "src.preparation.prepare:run"),
    ("transform", "src.transformation.transform:run"),
    ("features", "src.features.feature_store:build_features"),
    ("train", "src.training.train:run"),
    ("evaluate", "src.evaluation.evaluate:run"),
    ("report", "src.evaluation.report:run"),
]
FN = dict(STAGES)


def run_stage(name: str, config_path: str | None = None):
    """Import + run one stage, recording timing and status in reports/run_log.jsonl."""
    mod_name, fn = FN[name].rsplit(":", 1)
    cfg = load_config(config_path)
    t0, started = time.time(), datetime.now(timezone.utc).isoformat(timespec="seconds")
    status, err = "COMPLETED", None
    try:
        return getattr(importlib.import_module(mod_name), fn)(cfg)
    except Exception as e:  # noqa: BLE001
        status, err = "FAILED", repr(e)
        raise
    finally:
        with open(RUN_LOG, "a") as f:
            f.write(json.dumps({"stage": name, "status": status, "started": started,
                                "seconds": round(time.time() - t0, 2), "error": err}) + "\n")


def run_sequential(config_path: str | None = None):
    RUN_LOG.parent.mkdir(exist_ok=True)
    RUN_LOG.write_text("")
    for name, _ in STAGES:
        run_stage(name, config_path)


In [ ]:
%%writefile recommendation-pipeline/src/ingestion/__init__.py
# (package marker)


In [ ]:
%%writefile recommendation-pipeline/src/ingestion/synthetic.py
"""Synthetic movie-platform data generator (users, items, interactions) with injected data-quality problems.

The clean data has real taste structure (genre preferences + latent factors + popularity skew), so a
personalised model has something to learn. A controlled amount of dirt is then injected so the
validation stage has realistic work to do.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

GENRES = ["action", "comedy", "drama", "sci-fi", "romance", "thriller",
          "documentary", "animation", "horror", "fantasy"]
COUNTRIES = ["IN", "US", "UK", "DE", "FR", "BR", "CA", "AU"]
TS_FMT = "%Y-%m-%d %H:%M:%S"


def generate_clean(cfg: dict):
    d = cfg["data"]
    rng = np.random.default_rng(cfg["seed"])
    n_u, n_i, G, D = d["n_users"], d["n_items"], len(GENRES), 4
    start, end = pd.Timestamp(d["start_date"]), pd.Timestamp(d["end_date"])
    span = (end - start).total_seconds()

    # ---- items
    genre_w = rng.dirichlet(np.ones(G) * 3)
    n_g = rng.choice([1, 2, 3], size=n_i, p=[0.45, 0.40, 0.15])
    item_G = np.zeros((n_i, G))
    for i in range(n_i):
        item_G[i, rng.choice(G, size=n_g[i], replace=False, p=genre_w)] = 1
    pop = rng.lognormal(0, 1.1, n_i)
    quality = rng.normal(0, 0.45, n_i)
    q = rng.normal(0, 1, (n_i, D))
    items = pd.DataFrame({
        "item_id": np.arange(1, n_i + 1),
        "title": [f"Movie {i:04d}" for i in range(1, n_i + 1)],
        "category": ["|".join(GENRES[g] for g in np.flatnonzero(row)) for row in item_G],
        "release_year": rng.integers(1990, 2024, n_i),
    })

    # ---- users
    taste = rng.dirichlet(np.ones(G) * 0.35, n_u)
    p = rng.normal(0, 1, (n_u, D))
    ub = rng.normal(0, 0.35, n_u)
    signup = start + pd.to_timedelta(rng.uniform(0, 0.7, n_u) * span, unit="s")
    users = pd.DataFrame({
        "user_id": np.arange(1, n_u + 1),
        "age": rng.integers(16, 70, n_u),
        "gender": rng.choice(["m", "f", "other"], n_u, p=[0.48, 0.48, 0.04]),
        "country": rng.choice(COUNTRIES, n_u),
        "signup_date": signup.strftime("%Y-%m-%d"),
    })

    # ---- utility: genre affinity + latent taste
    gen_aff = (taste @ item_G.T) / item_G.sum(1)[None, :]
    lat = (p @ q.T) / np.sqrt(D)
    z_gen = (gen_aff - gen_aff.mean()) / gen_aff.std()
    z_lat = lat / lat.std()
    util = 0.9 * z_gen + 0.6 * z_lat
    rating = 3.4 + ub[:, None] + quality[None, :] + 0.55 * util + rng.normal(0, 0.5, util.shape)
    rating = np.clip(np.rint(rating), 1, 5).astype(int)

    # ---- interactions: selection depends on popularity AND taste
    sel = pop[None, :] * np.exp(0.8 * util)
    sel = sel / sel.sum(1, keepdims=True)
    counts = np.clip(rng.lognormal(np.log(d["mean_interactions_per_user"]), 0.7, n_u), 5, 250).astype(int)
    rows = []
    for u in range(n_u):
        chosen = rng.choice(n_i, size=min(counts[u], n_i), replace=False, p=sel[u])
        frac = rng.uniform(0, 1, len(chosen))
        t0 = signup[u]
        ts = t0 + pd.to_timedelta(frac * (end - t0).total_seconds(), unit="s")
        for c, t in zip(chosen, ts):
            rows.append((u + 1, c + 1, rating[u, c], t.floor("s")))
    inter = pd.DataFrame(rows, columns=["user_id", "item_id", "rating", "timestamp"])
    return users, items, inter


def inject_noise(users, items, inter, cfg):
    """Return object-typed frames with realistic problems injected (rates scale with data.dirty_rate)."""
    rng = np.random.default_rng(cfg["seed"] + 1)
    r = cfg["data"]["dirty_rate"]
    users, items, inter = users.astype(object), items.astype(object), inter.copy()
    inter["timestamp"] = inter["timestamp"].dt.strftime(TS_FMT)
    inter = inter.astype(object)

    def pick(df, frac):
        return rng.choice(df.index.to_numpy(), size=max(1, int(len(df) * frac)), replace=False)

    # ---- users
    users.loc[pick(users, r * 1.7), "age"] = np.nan                         # missing age
    bad_age = pick(users, r * 0.35)
    users.loc[bad_age, "age"] = rng.choice([-5, 0, 150, 999], len(bad_age))   # invalid age
    users.loc[pick(users, r), "country"] = np.nan
    users.loc[pick(users, r * 0.7), "gender"] = np.nan
    users.loc[pick(users, r * 0.2), "signup_date"] = "31/02/2024"            # malformed date
    users = pd.concat([users, users.loc[pick(users, r * 0.35)]], ignore_index=True)  # exact duplicates

    # ---- items
    items.loc[pick(items, r * 1.3), "title"] = np.nan
    items.loc[pick(items, r), "category"] = np.nan
    items.loc[pick(items, r * 0.2), "release_year"] = 2099
    items = pd.concat([items, items.loc[pick(items, r * 0.35)]], ignore_index=True)

    # ---- interactions
    signup_map = users.drop_duplicates("user_id").set_index("user_id")["signup_date"]
    inter.loc[pick(inter, r * 0.45), "user_id"] = "unknown"                  # wrong dtype
    idx = pick(inter, r * 0.8)
    inter.loc[idx, "rating"] = rng.choice([0, 6, -1, 10, 99], len(idx))      # out-of-range rating
    inter.loc[pick(inter, r * 0.35), "rating"] = np.nan                      # missing rating
    inter.loc[pick(inter, r * 0.1), "rating"] = "five"                       # non-numeric rating
    inter.loc[pick(inter, r * 0.3), "timestamp"] = "not-a-date"
    inter.loc[pick(inter, r * 0.3), "timestamp"] = "2024-13-45 99:00:00"
    inter.loc[pick(inter, r * 0.3), "timestamp"] = "2031-05-01 00:00:00"    # future timestamp
    idx = pick(inter, r * 0.5)
    inter.loc[idx, "user_id"] = rng.integers(90000, 99999, len(idx))         # orphan user
    idx = pick(inter, r * 0.5)
    inter.loc[idx, "item_id"] = rng.integers(90000, 99999, len(idx))         # orphan item
    # interaction before the user signed up
    for i in pick(inter, r * 0.17):
        s = signup_map.get(inter.at[i, "user_id"])
        if isinstance(s, str) and "/" not in s:
            inter.at[i, "timestamp"] = (pd.Timestamp(s) - pd.Timedelta(days=int(rng.integers(1, 60)))).strftime(TS_FMT)
    # legitimate re-ratings (same user+item, later timestamp, new rating) -> resolved in preparation
    re = inter.loc[pick(inter, r * 0.7)].copy()
    ts = pd.to_datetime(re["timestamp"], format=TS_FMT, errors="coerce")
    ok = ts.notna() & (ts < pd.Timestamp("2025-01-01"))
    new_ts = (ts + pd.to_timedelta(rng.integers(1, 30, len(re)), unit="D")).dt.strftime(TS_FMT)
    re["timestamp"] = new_ts.where(ok, re["timestamp"])
    re["rating"] = rng.integers(1, 6, len(re))
    inter = pd.concat([inter, re], ignore_index=True)
    inter = pd.concat([inter, inter.loc[pick(inter, r * 0.5)]], ignore_index=True)  # exact duplicates
    return users, items, inter.sample(frac=1.0, random_state=cfg["seed"]).reset_index(drop=True)


def generate_to_csv(cfg: dict, out_dir) -> dict:
    users, items, inter = generate_clean(cfg)
    u, i, x = inject_noise(users, items, inter, cfg)
    u.to_csv(out_dir / "users.csv", index=False)
    i.to_csv(out_dir / "items.csv", index=False)
    x.to_csv(out_dir / "interactions.csv", index=False)
    return {"users": len(u), "items": len(i), "interactions": len(x)}


In [ ]:
%%writefile recommendation-pipeline/src/ingestion/ingest.py
"""Stage 1 - Data ingestion.

Reads users / items / interactions from CSV (or generates documented synthetic data first),
loads everything as *strings* so no silent type coercion happens before validation, and writes an
ingestion log (row counts, columns, MD5 checksums) for lineage/reproducibility.
"""
from __future__ import annotations

import hashlib
import time
from pathlib import Path

import pandas as pd

from src.ingestion.synthetic import generate_to_csv
from src.utils import get_logger, write_json

log = get_logger("ingestion")
TABLES = ["users", "items", "interactions"]


def _md5(path: Path) -> str:
    return hashlib.md5(path.read_bytes()).hexdigest()


def load_raw(cfg: dict) -> dict[str, pd.DataFrame]:
    raw_dir = cfg["_paths"]["raw"]
    return {t: pd.read_csv(raw_dir / f"{t}.csv", dtype=str, keep_default_na=True) for t in TABLES}


def run(cfg: dict) -> dict:
    t0 = time.time()
    raw_dir = cfg["_paths"]["raw"]
    src = cfg["data"]["source"]
    if src == "synthetic":
        log.info("Generating synthetic data (seed=%s)", cfg["seed"])
        generate_to_csv(cfg, raw_dir)
    elif src == "csv":
        csv_dir = Path(cfg["data"]["csv_dir"])
        if csv_dir.resolve() != raw_dir.resolve():
            for t in TABLES:
                (raw_dir / f"{t}.csv").write_bytes((csv_dir / f"{t}.csv").read_bytes())
    else:
        raise ValueError(f"Unknown data.source: {src}")

    log_rows = {}
    for t, df in load_raw(cfg).items():
        f = raw_dir / f"{t}.csv"
        log_rows[t] = {"rows": len(df), "columns": list(df.columns), "md5": _md5(f),
                       "missing_cells": int(df.isna().sum().sum())}
        log.info("Ingested %-12s rows=%d cols=%d md5=%s", t, len(df), df.shape[1], log_rows[t]["md5"][:8])
    summary = {"source": src, "tables": log_rows, "seconds": round(time.time() - t0, 2)}
    write_json(summary, cfg["_paths"]["reports"] / "ingestion_log.json")
    return summary


In [ ]:
%%writefile recommendation-pipeline/src/validation/__init__.py
# (package marker)


In [ ]:
%%writefile recommendation-pipeline/src/validation/validate.py
"""Stage 2 - Data validation.

Rule-based checks (missing values, duplicates, data types, rating range, timestamp validity,
referential integrity). Each check has a severity:

  * error   -> the record is REJECTED and written to data/errors/<table>_errors.csv with its reasons
  * warning -> the record is KEPT and the issue is repaired in the preparation stage (e.g. impute age)

Outputs: validated typed Parquet tables, error datasets, reports/validation_report.{md,json}.
A quality gate aborts the pipeline if too large a share of any table is rejected.
"""
from __future__ import annotations

import time

import numpy as np
import pandas as pd

from src.ingestion.ingest import load_raw
from src.utils import get_logger, write_json

log = get_logger("validation")
TS_FMT = "%Y-%m-%d %H:%M:%S"


class DataQualityError(RuntimeError):
    pass


class _Checker:
    """Collects per-row reasons and per-check statistics for one table."""

    def __init__(self, table: str, df: pd.DataFrame):
        self.table, self.df = table, df
        self.err = pd.Series("", index=df.index, dtype=object)
        self.warn = pd.Series("", index=df.index, dtype=object)
        self.results: list[dict] = []

    def add(self, name: str, mask: pd.Series, severity: str, desc: str, n_checked: int | None = None):
        mask = mask.fillna(False).astype(bool)
        target = self.err if severity == "error" else self.warn
        target[mask] = target[mask] + name + ";"
        self.results.append({
            "table": self.table, "check": name, "severity": severity, "description": desc,
            "checked": int(n_checked if n_checked is not None else len(self.df)),
            "failed": int(mask.sum()),
            "action": "rejected -> errors dataset" if severity == "error" else "kept, repaired in preparation",
        })

    @property
    def error_mask(self) -> pd.Series:
        return self.err != ""


def _strip(s: pd.Series) -> pd.Series:
    return s.astype("object").where(s.isna(), s.astype(str).str.strip())


def _missing(s: pd.Series) -> pd.Series:
    return s.isna() | (_strip(s).astype(str) == "")


def _to_int(s: pd.Series) -> pd.Series:
    n = pd.to_numeric(_strip(s), errors="coerce")
    return n.where(n.notna() & (n == n.round()))


def _dup_checks(c: _Checker, key_col: str, key: pd.Series):
    exact = c.df.duplicated(keep="first")
    pk = key.notna() & key.duplicated(keep="first") & ~exact
    c.add("exact_duplicate_row", exact, "error", "Row is an exact copy of an earlier row")
    c.add("duplicate_primary_key", pk, "error", f"{key_col} repeated with different content")


# ----------------------------------------------------------------------------------------------
def validate_users(raw: pd.DataFrame, v: dict, now: pd.Timestamp):
    c = _Checker("users", raw)
    uid_missing, uid = _missing(raw.user_id), _to_int(raw.user_id)
    c.add("missing_user_id", uid_missing, "error", "user_id is empty")
    c.add("invalid_dtype_user_id", uid.isna() & ~uid_missing, "error", "user_id is not an integer")
    _dup_checks(c, "user_id", uid)
    age = pd.to_numeric(_strip(raw.age), errors="coerce")
    c.add("missing_age", _missing(raw.age), "warning", "age is empty -> imputed with median")
    c.add("age_out_of_range", age.notna() & ((age < v["age_min"]) | (age > v["age_max"])), "warning",
          f"age outside [{v['age_min']}, {v['age_max']}] -> set to missing, then imputed")
    c.add("missing_country", _missing(raw.country), "warning", "country is empty -> 'unknown'")
    c.add("missing_gender", _missing(raw.gender), "warning", "gender is empty -> 'unknown'")
    signup = pd.to_datetime(_strip(raw.signup_date), format="%Y-%m-%d", errors="coerce")
    c.add("invalid_signup_date", signup.isna() & ~_missing(raw.signup_date), "error", "signup_date is not YYYY-MM-DD")
    c.add("missing_signup_date", _missing(raw.signup_date), "error", "signup_date is empty")
    c.add("future_signup_date", signup > now, "error", "signup_date is after the reference date")
    good = ~c.error_mask
    clean = pd.DataFrame({
        "user_id": uid[good].astype("int64"), "age": age[good].astype("float64"),
        "gender": _strip(raw.gender)[good], "country": _strip(raw.country)[good],
        "signup_date": signup[good]}).reset_index(drop=True)
    return clean, c


def validate_items(raw: pd.DataFrame, v: dict, now: pd.Timestamp):
    c = _Checker("items", raw)
    iid_missing, iid = _missing(raw.item_id), _to_int(raw.item_id)
    c.add("missing_item_id", iid_missing, "error", "item_id is empty")
    c.add("invalid_dtype_item_id", iid.isna() & ~iid_missing, "error", "item_id is not an integer")
    _dup_checks(c, "item_id", iid)
    yr = pd.to_numeric(_strip(raw.release_year), errors="coerce")
    c.add("missing_title", _missing(raw.title), "warning", "title is empty -> placeholder title")
    c.add("missing_category", _missing(raw.category), "warning", "category is empty -> 'unknown'")
    c.add("release_year_out_of_range", yr.notna() & ((yr < v["release_year_min"]) | (yr > now.year)), "warning",
          "release_year outside plausible range -> set to missing, then imputed")
    good = ~c.error_mask
    clean = pd.DataFrame({
        "item_id": iid[good].astype("int64"), "title": _strip(raw.title)[good],
        "category": _strip(raw.category)[good], "release_year": yr[good].astype("float64")}).reset_index(drop=True)
    return clean, c


def validate_interactions(raw: pd.DataFrame, v: dict, now: pd.Timestamp,
                          users: pd.DataFrame, items: pd.DataFrame):
    c = _Checker("interactions", raw)
    uid_m, iid_m = _missing(raw.user_id), _missing(raw.item_id)
    uid, iid = _to_int(raw.user_id), _to_int(raw.item_id)
    c.add("missing_user_id", uid_m, "error", "user_id is empty")
    c.add("missing_item_id", iid_m, "error", "item_id is empty")
    c.add("invalid_dtype_user_id", uid.isna() & ~uid_m, "error", "user_id is not an integer")
    c.add("invalid_dtype_item_id", iid.isna() & ~iid_m, "error", "item_id is not an integer")
    # rating
    r_missing = _missing(raw.rating)
    rating = pd.to_numeric(_strip(raw.rating), errors="coerce")
    c.add("missing_rating", r_missing, "error", "rating is empty")
    c.add("invalid_dtype_rating", rating.isna() & ~r_missing, "error", "rating is not numeric")
    c.add("rating_out_of_range", rating.notna() & ((rating < v["rating_min"]) | (rating > v["rating_max"])),
          "error", f"rating outside [{v['rating_min']}, {v['rating_max']}]")
    # timestamp
    t_missing = _missing(raw.timestamp)
    ts = pd.to_datetime(_strip(raw.timestamp), format=TS_FMT, errors="coerce")
    c.add("missing_timestamp", t_missing, "error", "timestamp is empty")
    c.add("invalid_timestamp", ts.isna() & ~t_missing, "error", f"timestamp is not '{TS_FMT}'")
    c.add("future_timestamp", ts > now, "error", "timestamp is after the reference date")
    # duplicates
    c.add("exact_duplicate_row", raw.duplicated(keep="first"), "error", "Row is an exact copy of an earlier row")
    pair_dup = uid.notna() & iid.notna() & pd.DataFrame({"u": uid, "i": iid}).duplicated(keep=False)
    c.add("repeat_user_item_pair", pair_dup & ~raw.duplicated(keep=False), "warning",
          "same (user, item) rated more than once -> latest rating kept in preparation")
    # referential integrity (against the *validated* dimension tables)
    c.add("orphan_user_id", uid.notna() & ~uid.isin(users.user_id), "error", "user_id not found in validated users")
    c.add("orphan_item_id", iid.notna() & ~iid.isin(items.item_id), "error", "item_id not found in validated items")
    # temporal consistency
    signup = uid.map(users.set_index("user_id")["signup_date"])
    c.add("interaction_before_signup", ts.notna() & signup.notna() & (ts.dt.normalize() < signup),
          "error", "interaction happened before the user's signup_date")
    good = ~c.error_mask
    clean = pd.DataFrame({
        "user_id": uid[good].astype("int64"), "item_id": iid[good].astype("int64"),
        "rating": rating[good].astype("float64"), "timestamp": ts[good]}).reset_index(drop=True)
    return clean, c


# ----------------------------------------------------------------------------------------------
def _report_md(stats: dict, results: list[dict]) -> str:
    lines = ["# Data Validation Report", "",
             f"Reference date: `{stats['reference_date']}` | Quality gate: max {stats['max_error_rate']:.0%} rejected per table", "",
             "## Summary", "", "| Table | Records in | Valid | Rejected | Rejected % | Warnings (rows) |",
             "|---|---:|---:|---:|---:|---:|"]
    for t, s in stats["tables"].items():
        lines.append(f"| {t} | {s['records_in']:,} | {s['valid']:,} | {s['rejected']:,} | "
                     f"{s['rejected_pct']:.2f}% | {s['warning_rows']:,} |")
    lines += ["", "## Check results", "",
              "| Table | Check | Severity | Failed | Checked | Failed % | Description | Action |",
              "|---|---|---|---:|---:|---:|---|---|"]
    for r in results:
        pct = 100 * r["failed"] / max(r["checked"], 1)
        lines.append(f"| {r['table']} | `{r['check']}` | {r['severity']} | {r['failed']:,} | {r['checked']:,} | "
                     f"{pct:.2f}% | {r['description']} | {r['action']} |")
    lines += ["", "*Note:* a row can fail several checks, so per-check counts can sum to more than the rejected total.", ""]
    return "\n".join(lines)


def run(cfg: dict) -> dict:
    t0 = time.time()
    v, paths = cfg["validation"], cfg["_paths"]
    now = pd.Timestamp(v["reference_date"])
    raw = load_raw(cfg)

    users, cu = validate_users(raw["users"], v, now)
    items, ci = validate_items(raw["items"], v, now)
    inter, cx = validate_interactions(raw["interactions"], v, now, users, items)

    stats, results = {"reference_date": v["reference_date"], "max_error_rate": v["max_error_rate"], "tables": {}}, []
    for name, checker, clean in [("users", cu, users), ("items", ci, items), ("interactions", cx, inter)]:
        err_df = checker.df[checker.error_mask].copy()
        err_df["error_reasons"] = checker.err[checker.error_mask]
        err_df.to_csv(paths["errors"] / f"{name}_errors.csv", index=False)
        clean.to_parquet(paths["processed"] / f"validated_{name}.parquet", index=False)
        n_in, n_bad = len(checker.df), int(checker.error_mask.sum())
        stats["tables"][name] = {"records_in": n_in, "valid": len(clean), "rejected": n_bad,
                                 "rejected_pct": 100 * n_bad / n_in, "warning_rows": int((checker.warn != "").sum())}
        results += checker.results
        log.info("%-12s in=%d valid=%d rejected=%d (%.2f%%) warning_rows=%d", name, n_in, len(clean), n_bad,
                 100 * n_bad / n_in, stats["tables"][name]["warning_rows"])
        for r in checker.results:
            if r["failed"]:
                log.info("   [%s] %-28s failed=%d", r["severity"].upper(), r["check"], r["failed"])

    stats["checks"] = results
    stats["seconds"] = round(time.time() - t0, 2)
    write_json(stats, paths["reports"] / "validation_report.json")
    (paths["reports"] / "validation_report.md").write_text(_report_md(stats, results))

    worst = max(s["rejected_pct"] for s in stats["tables"].values()) / 100
    if worst > v["max_error_rate"]:
        raise DataQualityError(f"Quality gate failed: {worst:.1%} rejected > {v['max_error_rate']:.0%}")
    log.info("Quality gate passed (worst table rejected %.2f%% <= %.0f%%)", 100 * worst, 100 * v["max_error_rate"])
    return {"tables": stats["tables"], "seconds": stats["seconds"]}


In [ ]:
%%writefile recommendation-pipeline/src/preparation/__init__.py
# (package marker)


In [ ]:
%%writefile recommendation-pipeline/src/preparation/prepare.py
"""Stage 3 - Data preparation.

Cleans the validated tables (imputation, standardisation, de-duplication) and creates the
train/test datasets with a TIME-BASED split (no random shuffling -> no look-ahead leakage).
"""
from __future__ import annotations

import time

import numpy as np
import pandas as pd

from src.utils import get_logger, write_json

log = get_logger("preparation")


def clean_users(u: pd.DataFrame, v: dict) -> pd.DataFrame:
    u = u.copy()
    u.loc[(u.age < v["age_min"]) | (u.age > v["age_max"]), "age"] = np.nan   # invalid -> missing
    u["age"] = u["age"].fillna(u["age"].median()).round().astype("int64")
    u["gender"] = u["gender"].fillna("unknown").str.lower().str.strip().replace("", "unknown")
    u["country"] = u["country"].fillna("unknown").str.upper().str.strip().replace("", "UNKNOWN").replace("UNKNOWN", "unknown")
    u["age_group"] = pd.cut(u["age"], [0, 24, 34, 44, 54, 200], labels=["<25", "25-34", "35-44", "45-54", "55+"]).astype(str)
    return u.sort_values("user_id").reset_index(drop=True)


def clean_items(i: pd.DataFrame, v: dict, now: pd.Timestamp) -> pd.DataFrame:
    i = i.copy()
    i["title"] = i["title"].fillna("").str.strip()
    i.loc[i.title == "", "title"] = "Untitled item " + i.loc[i.title == "", "item_id"].astype(str)
    i["category"] = i["category"].fillna("unknown").str.lower().str.strip().replace("", "unknown")
    i["categories"] = i["category"].apply(lambda s: "|".join(sorted(set(x.strip() for x in s.split("|")))))
    i.loc[(i.release_year < v["release_year_min"]) | (i.release_year > now.year), "release_year"] = np.nan
    i["release_year"] = i["release_year"].fillna(i["release_year"].median()).astype("int64")
    return i.drop(columns="category").rename(columns={"categories": "category"}).sort_values("item_id").reset_index(drop=True)


def clean_interactions(x: pd.DataFrame) -> pd.DataFrame:
    x = x.sort_values(["timestamp", "user_id", "item_id"])
    before = len(x)
    x = x.drop_duplicates(["user_id", "item_id"], keep="last")        # latest rating per (user, item) wins
    log.info("De-duplicated (user,item) pairs: %d -> %d rows", before, len(x))
    return x.reset_index(drop=True)


def time_split(x: pd.DataFrame, test_fraction: float):
    cutoff = x["timestamp"].quantile(1 - test_fraction)
    return x[x.timestamp < cutoff].copy(), x[x.timestamp >= cutoff].copy(), cutoff


def run(cfg: dict) -> dict:
    t0 = time.time()
    p, v = cfg["_paths"], cfg["validation"]
    now = pd.Timestamp(v["reference_date"])
    users = clean_users(pd.read_parquet(p["processed"] / "validated_users.parquet"), v)
    items = clean_items(pd.read_parquet(p["processed"] / "validated_items.parquet"), v, now)
    inter = clean_interactions(pd.read_parquet(p["processed"] / "validated_interactions.parquet"))

    train, test, cutoff = time_split(inter, cfg["preparation"]["test_fraction"])
    test["user_in_train"] = test.user_id.isin(train.user_id)
    test["item_in_train"] = test.item_id.isin(train.item_id)
    warm = test.user_in_train & test.item_in_train

    users.to_parquet(p["processed"] / "users_clean.parquet", index=False)
    items.to_parquet(p["processed"] / "items_clean.parquet", index=False)
    inter.to_parquet(p["processed"] / "interactions_clean.parquet", index=False)
    train.to_parquet(p["processed"] / "train.parquet", index=False)
    test.to_parquet(p["processed"] / "test.parquet", index=False)

    meta = {
        "cutoff": str(cutoff), "train_rows": len(train), "test_rows": len(test),
        "test_rows_warm": int(warm.sum()), "test_rows_cold_user": int((~test.user_in_train).sum()),
        "test_rows_cold_item": int((~test.item_in_train).sum()),
        "train_range": [str(train.timestamp.min()), str(train.timestamp.max())],
        "test_range": [str(test.timestamp.min()), str(test.timestamp.max())],
        "n_users": len(users), "n_items": len(items), "n_interactions_clean": len(inter),
        "seconds": round(time.time() - t0, 2),
    }
    write_json(meta, p["reports"] / "split_meta.json")
    log.info("Time split at %s -> train=%d test=%d (warm-start test rows=%d)", cutoff, len(train), len(test), warm.sum())
    return meta


In [ ]:
%%writefile recommendation-pipeline/src/transformation/__init__.py
# (package marker)


In [ ]:
%%writefile recommendation-pipeline/src/transformation/transform.py
"""Stage 4 - Data transformation: derived datasets built from the TRAIN split only.

  * user_activity_summary  - per-user activity statistics
  * item_popularity        - per-item counts, mean rating, Bayesian rating, popularity rank
  * item_genre_matrix      - multi-hot item x genre matrix (content side-information)
  * user_item_matrix.npz   - sparse CSR rating matrix (+ matrix_index.json id <-> row/col mapping)
"""
from __future__ import annotations

import time

import numpy as np
import pandas as pd
import scipy.sparse as sp

from src.utils import get_logger, write_json

log = get_logger("transformation")


def item_genre_matrix(items: pd.DataFrame) -> pd.DataFrame:
    g = items.set_index("item_id")["category"].str.get_dummies(sep="|")
    g.columns = [f"genre_{c}" for c in g.columns]
    return g.astype("int8").sort_index()


def run(cfg: dict) -> dict:
    t0 = time.time()
    p = cfg["_paths"]
    train = pd.read_parquet(p["processed"] / "train.parquet")
    items = pd.read_parquet(p["processed"] / "items_clean.parquet")
    cutoff = pd.Timestamp(__import__("json").loads((p["reports"] / "split_meta.json").read_text())["cutoff"])
    prior = cfg["features"]["bayes_prior_count"]

    ua = train.groupby("user_id").agg(
        n_interactions=("item_id", "count"), n_distinct_items=("item_id", "nunique"),
        avg_rating=("rating", "mean"), first_interaction=("timestamp", "min"), last_interaction=("timestamp", "max"))
    ua["active_days"] = (ua.last_interaction.dt.normalize() - ua.first_interaction.dt.normalize()).dt.days + 1
    ua["interactions_per_active_day"] = ua.n_interactions / ua.active_days
    ua = ua.reset_index()

    gm = train.rating.mean()
    ip = train.groupby("item_id").agg(n_interactions=("user_id", "count"), n_users=("user_id", "nunique"),
                                      avg_rating=("rating", "mean"), rating_sum=("rating", "sum"))
    ip["bayes_avg_rating"] = (prior * gm + ip.rating_sum) / (prior + ip.n_interactions)
    ip["popularity_rank"] = ip.n_interactions.rank(ascending=False, method="min").astype(int)
    ip = ip.drop(columns="rating_sum").reset_index()

    genres = item_genre_matrix(items)
    user_ids = np.sort(train.user_id.unique())
    item_ids = np.sort(train.item_id.unique())
    u_idx = pd.Series(np.arange(len(user_ids)), index=user_ids)
    i_idx = pd.Series(np.arange(len(item_ids)), index=item_ids)
    M = sp.csr_matrix((train.rating.values, (u_idx[train.user_id].values, i_idx[train.item_id].values)),
                      shape=(len(user_ids), len(item_ids)))

    ua.to_parquet(p["processed"] / "user_activity_summary.parquet", index=False)
    ip.to_parquet(p["processed"] / "item_popularity.parquet", index=False)
    genres.reset_index().to_parquet(p["processed"] / "item_genre_matrix.parquet", index=False)
    sp.save_npz(p["processed"] / "user_item_matrix.npz", M)
    write_json({"user_ids": user_ids.tolist(), "item_ids": item_ids.tolist(), "global_mean": gm,
                "train_cutoff": str(cutoff)}, p["processed"] / "matrix_index.json")

    density = M.nnz / (M.shape[0] * M.shape[1])
    meta = {"matrix_shape": list(M.shape), "nnz": int(M.nnz), "density": density,
            "n_genres": genres.shape[1], "seconds": round(time.time() - t0, 2)}
    log.info("Interaction matrix %s nnz=%d density=%.3f", M.shape, M.nnz, density)
    return meta


In [ ]:
%%writefile recommendation-pipeline/src/features/__init__.py
# (package marker)


In [ ]:
%%writefile recommendation-pipeline/src/features/registry.py
"""Feature registry: single source of truth for every feature (name, entity, definition, source).

The feature catalog (reports/feature_catalog.md|csv) is generated from this list, so the documentation
can never drift from the code. All features are computed AS OF the train cutoff, from train data only.
"""
FEATURES = [
    # ---- user features (entity: user_id)
    ("user_avg_rating", "user", "float", "Mean rating given by the user", "mean(rating) over user's train interactions", "train.parquet"),
    ("user_rating_std", "user", "float", "Spread of the user's ratings (0 if one rating)", "std(rating), NaN->0", "train.parquet"),
    ("user_activity_count", "user", "int", "Number of interactions by the user", "count(interactions)", "train.parquet"),
    ("user_distinct_items", "user", "int", "Number of distinct items the user interacted with", "nunique(item_id)", "train.parquet"),
    ("user_days_since_last_interaction", "user", "float", "Recency: days between the user's last interaction and the train cutoff", "(cutoff - max(timestamp)) / 1 day", "train.parquet"),
    ("user_tenure_days", "user", "float", "Days between the user's first and last train interaction", "(max(ts) - min(ts)) / 1 day", "train.parquet"),
    ("user_days_since_signup", "user", "float", "Account age at the train cutoff", "(cutoff - signup_date) / 1 day", "users_clean.parquet"),
    ("user_preferred_category", "user", "str", "Genre with the highest weight in the user's profile", "argmax(genre_pref_*)", "train + item_genre_matrix"),
    ("user_preferred_category_share", "user", "float", "Share of the profile held by the preferred genre (taste concentration)", "max(genre_pref_*)", "train + item_genre_matrix"),
    ("genre_pref_<genre> (x11)", "user", "float", "Normalised genre preference profile (sums to 1); built from liked items (rating >= 4), falling back to all items", "L1-normalise(sum of item genre vectors)", "train + item_genre_matrix"),
    ("user_age / user_age_group / user_country", "user", "int/str", "Cleaned demographic attributes", "imputed + standardised in preparation", "users_clean.parquet"),
    # ---- item features (entity: item_id)
    ("item_popularity", "item", "int", "Number of train interactions with the item", "count(interactions)", "train.parquet"),
    ("item_unique_users", "item", "int", "Number of distinct users who interacted with the item", "nunique(user_id)", "train.parquet"),
    ("item_avg_rating", "item", "float", "Mean rating of the item", "mean(rating)", "train.parquet"),
    ("item_bayes_rating", "item", "float", "Mean rating shrunk toward the global mean (robust for rarely-rated items)", "(C*global_mean + sum(rating)) / (C + n), C=10", "train.parquet"),
    ("item_popularity_rank", "item", "int", "Rank by interaction count (1 = most popular)", "rank(-item_popularity)", "train.parquet"),
    ("item_popularity_pct", "item", "float", "Popularity percentile in [0,1]", "rank(item_popularity)/N", "train.parquet"),
    ("item_days_since_last_interaction", "item", "float", "Days between the item's last interaction and the train cutoff", "(cutoff - max(timestamp)) / 1 day", "train.parquet"),
    ("item_age_years", "item", "int", "Item age at the train cutoff", "cutoff.year - release_year", "items_clean.parquet"),
    ("item_n_categories", "item", "int", "Number of genres attached to the item", "sum(genre_*)", "item_genre_matrix"),
    ("item_primary_category", "item", "str", "First (alphabetical) genre of the item", "first(category)", "items_clean.parquet"),
    # ---- interaction / pair features (entity: user_id x item_id) - computed on demand
    ("user_item_category_affinity", "pair", "float", "How well the item's genres match the user's genre profile", "(genre_pref_u . genre_i) / n_categories_i", "user + item features"),
    ("item_avg_minus_user_avg", "pair", "float", "Item quality relative to the user's rating level", "item_bayes_rating - user_avg_rating", "user + item features"),
    ("item_popularity_pct_x_activity", "pair", "float", "Popularity percentile weighted by how active the user is (log scale)", "item_popularity_pct * log1p(user_activity_count)", "user + item features"),
]


In [ ]:
%%writefile recommendation-pipeline/src/features/feature_store.py
"""Stage 5 - Feature management: a small file-based offline feature store.

`build_features(cfg)` materialises user / item / pair features (as of the train cutoff, train data only).
`FeatureStore` is the SINGLE access path for those features, used both by training (models.py) and by
prediction (recommend.py), so train-time and serve-time features are identical (no training/serving skew).
"""
from __future__ import annotations

import json
import time

import numpy as np
import pandas as pd

from src.features.registry import FEATURES
from src.utils import get_logger, write_json

log = get_logger("features")


def genre_profiles(train: pd.DataFrame, item_genre: pd.DataFrame, threshold: float) -> pd.DataFrame:
    """User x genre preference profile (rows sum to 1). Uses liked items; falls back to all items."""
    G = item_genre.reindex(columns=item_genre.columns)
    liked = train[train.rating >= threshold]
    def prof(df):
        m = df.merge(G, left_on="item_id", right_index=True, how="left").groupby("user_id")[list(G.columns)].sum()
        return m
    p_all, p_liked = prof(train), prof(liked)
    p = p_liked.reindex(p_all.index).fillna(0.0)
    empty = p.sum(1) == 0
    p.loc[empty] = p_all.loc[empty]
    p = p.div(p.sum(1).replace(0, 1), axis=0)
    p.columns = [c.replace("genre_", "genre_pref_") for c in p.columns]
    return p


def build_features(cfg: dict) -> dict:
    t0 = time.time()
    P, prior, thr = cfg["_paths"], cfg["features"]["bayes_prior_count"], cfg["preparation"]["relevance_threshold"]
    train = pd.read_parquet(P["processed"] / "train.parquet")
    test = pd.read_parquet(P["processed"] / "test.parquet")
    users = pd.read_parquet(P["processed"] / "users_clean.parquet")
    items = pd.read_parquet(P["processed"] / "items_clean.parquet")
    genre = pd.read_parquet(P["processed"] / "item_genre_matrix.parquet").set_index("item_id")
    cutoff = pd.Timestamp(json.loads((P["processed"] / "matrix_index.json").read_text())["train_cutoff"])
    day = 86400.0

    # ---------------- user features
    g = train.groupby("user_id").agg(
        user_avg_rating=("rating", "mean"), user_rating_std=("rating", "std"),
        user_activity_count=("item_id", "count"), user_distinct_items=("item_id", "nunique"),
        _last=("timestamp", "max"), _first=("timestamp", "min"))
    g["user_rating_std"] = g["user_rating_std"].fillna(0.0)
    g["user_days_since_last_interaction"] = (cutoff - g["_last"]).dt.total_seconds() / day
    g["user_tenure_days"] = (g["_last"] - g["_first"]).dt.total_seconds() / day
    prof = genre_profiles(train, genre, thr)
    g = g.drop(columns=["_last", "_first"]).join(prof)
    pref_cols = list(prof.columns)
    g["user_preferred_category"] = prof.idxmax(axis=1).str.replace("genre_pref_", "", regex=False)
    g["user_preferred_category_share"] = prof.max(axis=1)
    u = users.set_index("user_id")
    g["user_days_since_signup"] = (cutoff - u.loc[g.index, "signup_date"]).dt.total_seconds() / day
    g["user_age"], g["user_age_group"], g["user_country"] = u.loc[g.index, "age"], u.loc[g.index, "age_group"], u.loc[g.index, "country"]
    user_feats = g.reset_index()

    # ---------------- item features
    gm = train.rating.mean()
    h = train.groupby("item_id").agg(
        item_popularity=("user_id", "count"), item_unique_users=("user_id", "nunique"),
        item_avg_rating=("rating", "mean"), _sum=("rating", "sum"), _last=("timestamp", "max"))
    h["item_bayes_rating"] = (prior * gm + h["_sum"]) / (prior + h["item_popularity"])
    h["item_popularity_rank"] = h["item_popularity"].rank(ascending=False, method="min").astype(int)
    h["item_popularity_pct"] = h["item_popularity"].rank(pct=True)
    h["item_days_since_last_interaction"] = (cutoff - h["_last"]).dt.total_seconds() / day
    it = items.set_index("item_id").loc[h.index]
    h["item_age_years"] = cutoff.year - it["release_year"]
    h["item_n_categories"] = genre.loc[h.index].sum(1)
    h["item_primary_category"] = it["category"].str.split("|").str[0]
    item_feats = h.drop(columns=["_sum", "_last"]).reset_index()

    user_feats.to_parquet(P["features"] / "user_features.parquet", index=False)
    item_feats.to_parquet(P["features"] / "item_features.parquet", index=False)
    write_json({"as_of": str(cutoff), "global_mean_rating": gm, "genre_pref_columns": pref_cols,
                "genre_columns": list(genre.columns)}, P["features"] / "feature_meta.json")

    # ---------------- pair features, materialised for train + test rows (test uses TRAIN-time features only)
    fs = FeatureStore(cfg)
    for name, df in [("train", train), ("test", test)]:
        fs.pair_features(df[["user_id", "item_id"]]).assign(rating=df["rating"].values).to_parquet(
            P["features"] / f"interaction_features_{name}.parquet", index=False)

    write_catalog(cfg, user_feats, item_feats)
    meta = {"as_of": str(cutoff), "n_user_features": user_feats.shape[1] - 1, "n_item_features": item_feats.shape[1] - 1,
            "n_users": len(user_feats), "n_items": len(item_feats), "seconds": round(time.time() - t0, 2)}
    log.info("Features built: users=%d (%d cols) items=%d (%d cols)", len(user_feats), meta["n_user_features"],
             len(item_feats), meta["n_item_features"])
    return meta


def write_catalog(cfg: dict, user_feats: pd.DataFrame, item_feats: pd.DataFrame) -> None:
    rep = cfg["_paths"]["reports"]
    cat = pd.DataFrame(FEATURES, columns=["feature", "entity", "dtype", "description", "definition", "source"])
    cat.to_csv(rep / "feature_catalog.csv", index=False)
    lines = ["# Feature Catalog", "",
             "Features are computed **as of the train cutoff using train data only** (no leakage) and are served through "
             "`src/features/feature_store.py::FeatureStore`, which is used by both training and prediction.", "",
             "| Feature | Entity | Type | Description | Definition | Source |", "|---|---|---|---|---|---|"]
    for r in FEATURES:
        lines.append("| `{}` | {} | {} | {} | `{}` | {} |".format(*r))
    lines += ["", "## Materialised tables (`data/features/`)", "",
              "| File | Rows | Columns | Key |", "|---|---:|---:|---|",
              f"| user_features.parquet | {len(user_feats):,} | {user_feats.shape[1]} | user_id |",
              f"| item_features.parquet | {len(item_feats):,} | {item_feats.shape[1]} | item_id |",
              "| interaction_features_train/test.parquet | one row per interaction | 3 + rating | (user_id, item_id) |", ""]
    (rep / "feature_catalog.md").write_text("\n".join(lines))


class FeatureStore:
    """Read-side API over the materialised features. Used for training AND prediction."""

    def __init__(self, cfg: dict):
        d = cfg["_paths"]["features"]
        self.user = pd.read_parquet(d / "user_features.parquet").set_index("user_id")
        self.item = pd.read_parquet(d / "item_features.parquet").set_index("item_id")
        meta = json.loads((d / "feature_meta.json").read_text())
        self.meta = meta
        self.pref_cols, self.genre_cols = meta["genre_pref_columns"], meta["genre_columns"]
        genre = pd.read_parquet(cfg["_paths"]["processed"] / "item_genre_matrix.parquet").set_index("item_id")
        self.item_genre = genre[self.genre_cols]

    def get_user_features(self, user_ids) -> pd.DataFrame:
        return self.user.reindex(user_ids)

    def get_item_features(self, item_ids) -> pd.DataFrame:
        return self.item.reindex(item_ids)

    def user_profile_matrix(self, user_ids) -> np.ndarray:
        return self.user.reindex(user_ids)[self.pref_cols].fillna(0.0).to_numpy(float)

    def item_genre_matrix(self, item_ids) -> np.ndarray:
        return self.item_genre.reindex(item_ids).fillna(0).to_numpy(float)

    def affinity_matrix(self, user_ids, item_ids) -> np.ndarray:
        """Dense user x item category affinity: (profile_u . genre_i) / n_categories_i."""
        G = self.item_genre_matrix(item_ids)
        return (self.user_profile_matrix(user_ids) @ G.T) / np.maximum(G.sum(1), 1)[None, :]

    def pair_features(self, pairs: pd.DataFrame) -> pd.DataFrame:
        """Interaction features for (user_id, item_id) pairs - identical code path for train and serving."""
        uf, itf = self.get_user_features(pairs.user_id.values), self.get_item_features(pairs.item_id.values)
        P = uf[self.pref_cols].fillna(0.0).to_numpy(float)
        G = self.item_genre.reindex(pairs.item_id.values).fillna(0).to_numpy(float)
        aff = (P * G).sum(1) / np.maximum(G.sum(1), 1)
        gmean = self.meta["global_mean_rating"]
        return pd.DataFrame({
            "user_id": pairs.user_id.values, "item_id": pairs.item_id.values,
            "user_item_category_affinity": aff,
            "item_avg_minus_user_avg": itf["item_bayes_rating"].fillna(gmean).values - uf["user_avg_rating"].fillna(gmean).values,
            "item_popularity_pct_x_activity": itf["item_popularity_pct"].fillna(0).values * np.log1p(uf["user_activity_count"].fillna(0).values),
        })


In [ ]:
%%writefile recommendation-pipeline/src/training/__init__.py
# (package marker)


In [ ]:
%%writefile recommendation-pipeline/src/training/models.py
"""Recommender models. All share the same interface so training / evaluation are model-agnostic:

    score(user_idx)               -> (n_users, n_items) ranking scores (higher = better)
    predict_rating(u_idx, i_idx)  -> predicted ratings, or None if the model is ranking-only

Row / column indices refer to the train interaction matrix (see matrix_index.json).
"""
from __future__ import annotations

import numpy as np
import scipy.sparse as sp


def _zscore_rows(a: np.ndarray) -> np.ndarray:
    return (a - a.mean(1, keepdims=True)) / (a.std(1, keepdims=True) + 1e-9)


# ------------------------------------------------------------------------------------------ ALS
def _als_side(C: sp.csr_matrix, Y: np.ndarray, reg: float, implicit: bool, R: sp.csr_matrix | None = None):
    """Solve one side of ALS. implicit=True: Hu-Koren-Volinsky weighted ALS; else ridge on explicit residuals."""
    n, f = C.shape[0], Y.shape[1]
    out = np.zeros((n, f))
    eye = np.eye(f)
    YtY = Y.T @ Y
    for u in range(n):
        s, e = C.indptr[u], C.indptr[u + 1]
        if s == e:
            continue
        idx, vals = C.indices[s:e], C.data[s:e]
        Yu = Y[idx]
        if implicit:   # vals = confidence - 1 ; preference = 1 on observed entries
            A = YtY + (Yu.T * vals) @ Yu + reg * eye
            b = (Yu * (1.0 + vals)[:, None]).sum(0)
        else:          # vals = residual rating ; weighted-lambda regularisation
            A = Yu.T @ Yu + reg * (e - s) * eye
            b = Yu.T @ vals
        out[u] = np.linalg.solve(A, b)
    return out


class PopularityRecommender:
    """Baseline: every user gets the globally most popular items (non-personalised)."""
    name = "popularity"

    def fit(self, M, user_ids, item_ids, fs, cfg):
        f = fs.get_item_features(item_ids)
        self.pop = f["item_popularity"].to_numpy(float)
        self.item_rating = f["item_bayes_rating"].to_numpy(float)
        return self

    def params(self):
        return {"strategy": "item_popularity (train interaction count)", "rating_predictor": "item Bayesian mean rating"}

    def score(self, user_idx):
        return np.tile(self.pop, (len(user_idx), 1))

    def predict_rating(self, u_idx, i_idx):
        return np.clip(self.item_rating[i_idx], 1, 5)


class MFRecommender:
    """Personalised collaborative filtering via matrix factorisation (pure NumPy).

    * ranking  : implicit-feedback ALS (every interaction = positive signal, confidence grows with rating)
    * rating   : explicit biased MF  r_hat = mu + b_u + b_i + p_u . q_i  (ALS on bias residuals)
    """
    name = "als_mf"

    def __init__(self, seed: int = 42):
        self.seed = seed

    def fit(self, M: sp.csr_matrix, user_ids, item_ids, fs, cfg):
        a, b = cfg["training"]["als"], cfg["training"]["biased_mf"]
        rng = np.random.default_rng(self.seed)
        self.a, self.b = a, b
        # ---- implicit ALS for ranking
        C = M.copy().astype(float)
        C.data = a["alpha"] * C.data / 5.0
        CT = C.T.tocsr()
        X = rng.normal(0, 0.01, (M.shape[0], a["factors"]))
        Y = rng.normal(0, 0.01, (M.shape[1], a["factors"]))
        for _ in range(a["iterations"]):
            X = _als_side(C, Y, a["reg"], True)
            Y = _als_side(CT, X, a["reg"], True)
        self.X, self.Y = X, Y
        # ---- explicit biased MF for rating prediction
        R = M.tocsr().astype(float)
        self.mu = R.data.mean()
        cnt_i = np.diff(R.tocsc().indptr)
        bi = np.asarray(R.sum(0)).ravel() - self.mu * cnt_i
        self.bi = bi / (cnt_i + b["bias_reg"])
        cnt_u = np.diff(R.indptr)
        rowsum = np.asarray(R.sum(1)).ravel()
        resid_i = sp.csr_matrix((self.bi[R.indices], R.indices, R.indptr), shape=R.shape).sum(1).A1
        self.bu = (rowsum - self.mu * cnt_u - resid_i) / (cnt_u + b["bias_reg"])
        E = R.copy()
        E.data = R.data - self.mu - self.bu[np.repeat(np.arange(R.shape[0]), cnt_u)] - self.bi[R.indices]
        ET = E.T.tocsr()
        P = rng.normal(0, 0.1, (R.shape[0], b["factors"]))
        Q = rng.normal(0, 0.1, (R.shape[1], b["factors"]))
        for _ in range(b["iterations"]):
            P = _als_side(E, Q, b["reg"], False)
            Q = _als_side(ET, P, b["reg"], False)
        self.P, self.Q = P, Q
        return self

    def params(self):
        return {**{f"als_{k}": v for k, v in self.a.items()}, **{f"bmf_{k}": v for k, v in self.b.items()}}

    def score(self, user_idx):
        return self.X[user_idx] @ self.Y.T

    def predict_rating(self, u_idx, i_idx):
        r = self.mu + self.bu[u_idx] + self.bi[i_idx] + (self.P[u_idx] * self.Q[i_idx]).sum(1)
        return np.clip(r, 1, 5)


class HybridRecommender:
    """Hybrid: z-scored blend of CF score + content (category-affinity feature) + popularity prior.

    score = z(ALS) + w_content * z(affinity) + w_pop * z(log1p(popularity))
    Affinity and popularity come from the FeatureStore, so train-time and serve-time features match.
    Ranking-only (no rating prediction).
    """
    name = "hybrid"

    def __init__(self, base: MFRecommender, w_content: float = 0.3, w_pop: float = 0.1):
        self.base, self.w_content, self.w_pop = base, w_content, w_pop

    def fit(self, M, user_ids, item_ids, fs, cfg):
        self.profiles = fs.user_profile_matrix(user_ids)                 # (n_users, n_genres)
        self.G = fs.item_genre_matrix(item_ids)                          # (n_items, n_genres)
        self.log_pop = np.log1p(fs.get_item_features(item_ids)["item_popularity"].to_numpy(float))
        return self

    def params(self):
        return {"w_cf": 1.0, "w_content": self.w_content, "w_pop": self.w_pop, "base_model": self.base.name}

    def score(self, user_idx):
        aff = (self.profiles[user_idx] @ self.G.T) / np.maximum(self.G.sum(1), 1)[None, :]
        s = _zscore_rows(self.base.score(user_idx))
        s = s + self.w_content * _zscore_rows(aff)
        s = s + self.w_pop * _zscore_rows(np.tile(self.log_pop, (len(user_idx), 1)))
        return s

    def predict_rating(self, u_idx, i_idx):
        return None


In [ ]:
%%writefile recommendation-pipeline/src/training/train.py
"""Stage 6 - Model training with MLflow experiment tracking.

Trains (1) popularity baseline, (2) ALS matrix-factorisation CF, (3) hybrid CF + content + popularity.
The hybrid blend weights are tuned on a validation slice carved from the END of the training period
(time-based, no test leakage). Parameters, timings and artefacts are logged to MLflow; run ids are saved
to models/run_registry.json so the evaluation stage can attach its metrics to the same runs.
"""
from __future__ import annotations

import itertools
import pickle
import time

import mlflow
import numpy as np
import pandas as pd
import scipy.sparse as sp

from src.evaluation.evaluate import build_eval_set, evaluate_ranking
from src.features.feature_store import FeatureStore, genre_profiles
from src.training.models import HybridRecommender, MFRecommender, PopularityRecommender
from src.utils import get_logger, read_json, write_json

log = get_logger("training")


def _tune_hybrid(cfg, fs, train, uids, iids, u_map, i_map, G):
    """Grid-search hybrid weights on a time-based validation slice of the training data."""
    h, thr, pk = cfg["training"]["hybrid"], cfg["preparation"]["relevance_threshold"], cfg["evaluation"]["primary_k"]
    cut = train.timestamp.quantile(1 - h["val_fraction"])
    sub, val = train[train.timestamp < cut], train[train.timestamp >= cut]
    M_sub = sp.csr_matrix((sub.rating.values, (u_map[sub.user_id].values, i_map[sub.item_id].values)),
                          shape=(len(uids), len(iids)))
    val = val[val.user_id.isin(sub.user_id) & val.item_id.isin(sub.item_id)]
    _, ui, rel = build_eval_set(val, u_map, i_map, thr)
    base = MFRecommender(cfg["seed"]).fit(M_sub, uids, iids, fs, cfg)
    profiles = genre_profiles(sub, fs.item_genre, thr).reindex(uids).fillna(0.0).to_numpy(float)
    log_pop = np.log1p(np.bincount(i_map[sub.item_id].values, minlength=len(iids)).astype(float))
    grid = []
    for wc, wp in itertools.product(h["w_content_grid"], h["w_pop_grid"]):
        hyb = HybridRecommender(base, wc, wp)
        hyb.profiles, hyb.G, hyb.log_pop = profiles, fs.item_genre_matrix(iids), log_pop
        r = evaluate_ranking(hyb, M_sub, ui, rel, G, [pk])
        grid.append({"w_content": wc, "w_pop": wp, f"val_ndcg@{pk}": r[f"ndcg@{pk}"], f"val_diversity@{pk}": r[f"diversity@{pk}"]})
        log.info("  tune w_content=%.2f w_pop=%.2f -> val NDCG@%d=%.4f", wc, wp, pk, r[f"ndcg@{pk}"])
    best = max(grid, key=lambda g: g[f"val_ndcg@{pk}"])
    return best, grid


def run(cfg: dict) -> dict:
    t0 = time.time()
    P = cfg["_paths"]
    M = sp.load_npz(P["processed"] / "user_item_matrix.npz").tocsr()
    idx = read_json(P["processed"] / "matrix_index.json")
    uids, iids = np.array(idx["user_ids"]), np.array(idx["item_ids"])
    u_map, i_map = pd.Series(np.arange(len(uids)), index=uids), pd.Series(np.arange(len(iids)), index=iids)
    train = pd.read_parquet(P["processed"] / "train.parquet")
    fs = FeatureStore(cfg)
    G = fs.item_genre_matrix(iids)

    mlflow.set_tracking_uri(cfg["mlflow"]["tracking_uri"])
    mlflow.set_experiment(cfg["mlflow"]["experiment_name"])   # artifacts default to ./mlruns (run from project root)

    best, grid = None, None
    if cfg["training"]["hybrid"]["tune"]:
        log.info("Tuning hybrid weights on a time-based validation slice")
        best, grid = _tune_hybrid(cfg, fs, train, uids, iids, u_map, i_map, G)
        log.info("Best hybrid weights: %s", best)
    else:
        best = {"w_content": 0.3, "w_pop": 0.1}

    common = {"seed": cfg["seed"], "n_users": len(uids), "n_items": len(iids), "n_train_interactions": int(M.nnz),
              "train_cutoff": idx["train_cutoff"], "feature_store_as_of": fs.meta["as_of"]}
    registry = {"models": {}}
    base_mf = None
    for name in ["popularity", "als_mf", "hybrid"]:
        ts = time.time()
        if name == "popularity":
            model = PopularityRecommender().fit(M, uids, iids, fs, cfg)
        elif name == "als_mf":
            model = base_mf = MFRecommender(cfg["seed"]).fit(M, uids, iids, fs, cfg)
        else:
            model = HybridRecommender(base_mf, best["w_content"], best["w_pop"]).fit(M, uids, iids, fs, cfg)
        secs = time.time() - ts
        path = P["models"] / f"{name}.pkl"
        pickle.dump(model, open(path, "wb"))
        with mlflow.start_run(run_name=name) as run:
            mlflow.log_params({**common, **{k: str(v) for k, v in model.params().items()}})
            mlflow.set_tags({"model_type": name, "stage": "training"})
            mlflow.log_metric("train_seconds", secs)
            mlflow.log_artifact(str(path))
            mlflow.log_artifact(str(P["reports"] / "feature_catalog.md"))
            if name == "hybrid" and grid:
                write_json({"best": best, "grid": grid}, P["models"] / "hybrid_tuning.json")
                mlflow.log_artifact(str(P["models"] / "hybrid_tuning.json"))
                pk = cfg['evaluation']['primary_k']
                mlflow.log_metric(f"val_ndcg_at_{pk}", best[f"val_ndcg@{pk}"])
            registry["models"][name] = {"run_id": run.info.run_id, "file": path.name, "train_seconds": round(secs, 2)}
        log.info("Trained %-10s in %.1fs (MLflow run %s)", name, secs, run.info.run_id[:8])
    write_json(registry, P["models"] / "run_registry.json")
    return {"models": registry["models"], "best_hybrid": best, "seconds": round(time.time() - t0, 2)}


In [ ]:
%%writefile recommendation-pipeline/src/evaluation/__init__.py
# (package marker)


In [ ]:
%%writefile recommendation-pipeline/src/evaluation/metrics.py
"""Ranking and rating metrics (binary relevance)."""
from __future__ import annotations

import numpy as np


def top_k(scores: np.ndarray, seen_mask: np.ndarray, k: int) -> np.ndarray:
    """Top-k item indices per row, never recommending items the user already interacted with in train."""
    s = np.where(seen_mask, -np.inf, scores)
    part = np.argpartition(-s, k - 1, axis=1)[:, :k]
    order = np.argsort(-np.take_along_axis(s, part, 1), axis=1)
    return np.take_along_axis(part, order, 1)


def ranking_metrics(topk: np.ndarray, relevant: list[set], k: int) -> dict:
    prec, rec, ndcg = [], [], []
    disc = 1.0 / np.log2(np.arange(2, k + 2))
    for row, rel in zip(topk, relevant):
        hits = np.array([i in rel for i in row[:k]], dtype=float)
        prec.append(hits.sum() / k)
        rec.append(hits.sum() / len(rel))
        dcg = (hits * disc).sum()
        idcg = disc[: min(len(rel), k)].sum()
        ndcg.append(dcg / idcg)
    return {f"precision@{k}": float(np.mean(prec)), f"recall@{k}": float(np.mean(rec)), f"ndcg@{k}": float(np.mean(ndcg))}


def coverage(topk: np.ndarray, n_items: int) -> float:
    """Share of the catalogue that appears in at least one user's recommendation list."""
    return len(np.unique(topk)) / n_items


def diversity(topk: np.ndarray, item_genre: np.ndarray) -> float:
    """Mean intra-list diversity = 1 - mean pairwise cosine similarity of genre vectors in each list."""
    V = item_genre / np.maximum(np.linalg.norm(item_genre, axis=1, keepdims=True), 1e-9)
    k = topk.shape[1]
    out = []
    for row in topk:
        S = V[row] @ V[row].T
        out.append(1.0 - (S.sum() - np.trace(S)) / (k * (k - 1)))
    return float(np.mean(out))


def rating_errors(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    e = y_true - y_pred
    return {"rmse": float(np.sqrt(np.mean(e ** 2))), "mae": float(np.mean(np.abs(e)))}


In [ ]:
%%writefile recommendation-pipeline/src/evaluation/evaluate.py
"""Stage 7 - Model evaluation (Precision@K, Recall@K, NDCG@K, Coverage, Diversity, RMSE, MAE).

Protocol: warm-start users with >=1 relevant (rating >= threshold) item in the test period. Items already
seen in train are excluded from recommendations. Results are written to reports/ and logged to MLflow.
"""
from __future__ import annotations

import json
import pickle
import time

import mlflow
import numpy as np
import pandas as pd
import scipy.sparse as sp

from src.evaluation import metrics as m
from src.utils import get_logger, read_json, write_json

log = get_logger("evaluation")


def build_eval_set(df: pd.DataFrame, u_map: pd.Series, i_map: pd.Series, thr: float):
    """df: interaction rows (already warm). Returns (user ids, user row idx, list of relevant item-idx sets)."""
    rel = df[df.rating >= thr].groupby("user_id").item_id.apply(lambda s: set(i_map[s].values))
    users = np.array(rel.index)
    return users, u_map[users].values, list(rel.values)


def evaluate_ranking(model, M, user_idx, relsets, item_genre, ks) -> dict:
    scores = model.score(user_idx)
    seen = M[user_idx].toarray() > 0
    out = {}
    for k in ks:
        tk = m.top_k(scores, seen, k)
        out.update(m.ranking_metrics(tk, relsets, k))
        out[f"coverage@{k}"] = m.coverage(tk, M.shape[1])
        out[f"diversity@{k}"] = m.diversity(tk, item_genre)
    return out


def run(cfg: dict) -> dict:
    t0 = time.time()
    P, ev, thr = cfg["_paths"], cfg["evaluation"], cfg["preparation"]["relevance_threshold"]
    ks, pk = ev["k_values"], ev["primary_k"]
    M = sp.load_npz(P["processed"] / "user_item_matrix.npz").tocsr()
    idx = read_json(P["processed"] / "matrix_index.json")
    uids, iids = np.array(idx["user_ids"]), np.array(idx["item_ids"])
    u_map, i_map = pd.Series(np.arange(len(uids)), index=uids), pd.Series(np.arange(len(iids)), index=iids)
    test = pd.read_parquet(P["processed"] / "test.parquet")
    test = test[test.user_in_train & test.item_in_train]
    genre = pd.read_parquet(P["processed"] / "item_genre_matrix.parquet").set_index("item_id")
    G = genre.reindex(iids).to_numpy(float)
    items = pd.read_parquet(P["processed"] / "items_clean.parquet").set_index("item_id")

    eval_users, ui, relsets = build_eval_set(test, u_map, i_map, thr)
    tu, ti, y = u_map[test.user_id].values, i_map[test.item_id].values, test.rating.values
    log.info("Evaluating on %d users / %d warm test interactions (%d relevant users)", len(eval_users), len(test), len(ui))

    registry = read_json(P["models"] / "run_registry.json")
    mlflow.set_tracking_uri(cfg["mlflow"]["tracking_uri"])
    rows, sample_recs = [], []
    for name, info in registry["models"].items():
        model = pickle.load(open(P["models"] / info["file"], "rb"))
        res = evaluate_ranking(model, M, ui, relsets, G, ks)
        pred = model.predict_rating(tu, ti)
        res.update(m.rating_errors(y, pred) if pred is not None else {"rmse": np.nan, "mae": np.nan})
        res["model"] = name
        rows.append(res)
        # a few human-readable recommendations
        top = m.top_k(model.score(ui[:3]), M[ui[:3]].toarray() > 0, 5)
        for u, rec in zip(eval_users[:3], top):
            sample_recs.append({"model": name, "user_id": u, "recommended": "; ".join(
                f"{iids[j]} ({items.loc[iids[j], 'title']}, {items.loc[iids[j], 'category']})" for j in rec)})
        with mlflow.start_run(run_id=info["run_id"]):
            mlflow.log_metrics({k.replace("@", "_at_"): float(v) for k, v in res.items() if k != "model" and not pd.isna(v)})
        log.info("%-11s P@%d=%.4f R@%d=%.4f NDCG@%d=%.4f cov=%.3f div=%.3f", name, pk, res[f"precision@{pk}"], pk,
                 res[f"recall@{pk}"], pk, res[f"ndcg@{pk}"], res[f"coverage@{pk}"], res[f"diversity@{pk}"])

    df = pd.DataFrame(rows).set_index("model")
    cols = [f"{c}@{k}" for k in ks for c in ["precision", "recall", "ndcg", "coverage", "diversity"]] + ["rmse", "mae"]
    df = df[cols]
    df.to_csv(P["reports"] / "evaluation_results.csv")
    write_json({"protocol": {"eval_users": len(eval_users), "warm_test_rows": len(test), "relevance_threshold": thr,
                             "k_values": ks, "catalog_size": len(iids)}, "results": df.reset_index().to_dict("records")},
               P["reports"] / "evaluation_results.json")
    pd.DataFrame(sample_recs).to_csv(P["reports"] / "sample_recommendations.csv", index=False)
    (P["reports"] / "evaluation_results.md").write_text(_md(df, ks, pk, len(eval_users), thr, len(iids)))
    _plot(df, pk, P["reports"] / "model_comparison.png")
    # attach the artefacts to every model's run
    for info in registry["models"].values():
        with mlflow.start_run(run_id=info["run_id"]):
            for f in ["evaluation_results.csv", "evaluation_results.md", "model_comparison.png"]:
                mlflow.log_artifact(str(P["reports"] / f))
    return {"results": df.reset_index().to_dict("records"), "eval_users": len(eval_users), "seconds": round(time.time() - t0, 2)}


def _md(df, ks, pk, n_users, thr, n_items) -> str:
    L = ["# Model Evaluation Results", "",
         f"Evaluation users: **{n_users}** (warm-start, >=1 test item rated >= {thr:g}); catalogue = {n_items} items; "
         "train-seen items excluded from recommendations; time-based split.", ""]
    for k in ks:
        L += [f"## K = {k}", "", "| Model | Precision@K | Recall@K | NDCG@K | Coverage | Diversity |", "|---|---:|---:|---:|---:|---:|"]
        for n, r in df.iterrows():
            L.append(f"| {n} | {r[f'precision@{k}']:.4f} | {r[f'recall@{k}']:.4f} | {r[f'ndcg@{k}']:.4f} | "
                     f"{r[f'coverage@{k}']:.3f} | {r[f'diversity@{k}']:.3f} |")
        L.append("")
    L += ["## Rating prediction (all warm test interactions)", "", "| Model | RMSE | MAE |", "|---|---:|---:|"]
    for n, r in df.iterrows():
        L.append(f"| {n} | " + (f"{r['rmse']:.4f} | {r['mae']:.4f} |" if not pd.isna(r["rmse"]) else "n/a (ranking-only) | n/a |"))
    L += ["", "*Diversity* = 1 - mean pairwise cosine similarity of genre vectors inside a list (higher = more varied). "
          "*Coverage* = share of the catalogue appearing in at least one user's list.", ""]
    return "\n".join(L)


def _plot(df, pk, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    metrics = [f"precision@{pk}", f"recall@{pk}", f"ndcg@{pk}", f"coverage@{pk}", f"diversity@{pk}"]
    fig, axes = plt.subplots(1, len(metrics), figsize=(14, 3.2))
    colors = ["#9aa5b1", "#2f6fb2", "#d9822b"]
    for ax, met in zip(axes, metrics):
        ax.bar(df.index, df[met], color=colors[: len(df)])
        ax.set_title(met.upper().replace("@", "@"), fontsize=10)
        ax.tick_params(axis="x", labelrotation=25, labelsize=8)
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
        for j, v in enumerate(df[met]):
            ax.text(j, v, f"{v:.3f}", ha="center", va="bottom", fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


In [ ]:
%%writefile recommendation-pipeline/src/evaluation/report.py
"""Final pipeline task: collect stage outputs into reports/pipeline_summary.json."""
from __future__ import annotations

import mlflow

from src.utils import read_json, write_json


def run(cfg: dict) -> dict:
    R = cfg["_paths"]["reports"]
    summary = {
        "ingestion": read_json(R / "ingestion_log.json"),
        "validation": {t: s for t, s in read_json(R / "validation_report.json")["tables"].items()},
        "split": read_json(R / "split_meta.json"),
        "evaluation": read_json(R / "evaluation_results.json"),
    }
    write_json(summary, R / "pipeline_summary.json")
    # portable export of the MLflow experiment (params + metrics of every run)
    mlflow.set_tracking_uri(cfg["mlflow"]["tracking_uri"])
    runs = mlflow.search_runs(experiment_names=[cfg["mlflow"]["experiment_name"]])
    keep = [c for c in runs.columns if c.startswith(("params.", "metrics.", "tags.mlflow.runName"))]
    runs[["run_id", *keep]].to_csv(R / "mlflow_runs_summary.csv", index=False)
    return {"summary_file": "reports/pipeline_summary.json"}


In [ ]:
%%writefile recommendation-pipeline/dags/__init__.py
# (package marker)


In [ ]:
%%writefile recommendation-pipeline/dags/recommendation_flow.py
"""Stage 8 - Orchestration with Prefect.

DAG:  ingest -> validate -> prepare -> transform -> features -> train -> evaluate -> report

Each stage is a Prefect task; dependencies are declared with `wait_for`, so a stage only starts after its
upstream stage succeeded and the flow stops at the first failure (e.g. the data-quality gate).
Every stage logs start/end/status to reports/run_log.jsonl (used for the run-summary figure).

Run once:        python run_pipeline.py
Schedule daily:  python dags/recommendation_flow.py --serve        (cron: 02:00 every day)
"""
from __future__ import annotations

import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from prefect import flow, task  # noqa: E402

from src.pipeline import RUN_LOG, run_stage  # noqa: E402


@task(name="ingest", retries=1, retry_delay_seconds=2)
def ingest(config_path=None):
    return run_stage("ingest", config_path)


@task(name="validate")          # no retries: a failed quality gate is deterministic
def validate(config_path=None):
    return run_stage("validate", config_path)


@task(name="prepare", retries=1, retry_delay_seconds=2)
def prepare(config_path=None):
    return run_stage("prepare", config_path)


@task(name="transform", retries=1, retry_delay_seconds=2)
def transform(config_path=None):
    return run_stage("transform", config_path)


@task(name="features", retries=1, retry_delay_seconds=2)
def features(config_path=None):
    return run_stage("features", config_path)


@task(name="train")
def train(config_path=None):
    return run_stage("train", config_path)


@task(name="evaluate")
def evaluate(config_path=None):
    return run_stage("evaluate", config_path)


@task(name="report")
def report(config_path=None):
    return run_stage("report", config_path)


@flow(name="recommendation-pipeline", log_prints=True)
def recommendation_pipeline(config_path: str | None = None):
    RUN_LOG.parent.mkdir(exist_ok=True)
    RUN_LOG.write_text("")                                   # fresh log per run
    a = ingest.submit(config_path)
    b = validate.submit(config_path, wait_for=[a])
    c = prepare.submit(config_path, wait_for=[b])
    d = transform.submit(config_path, wait_for=[c])
    e = features.submit(config_path, wait_for=[d])
    f = train.submit(config_path, wait_for=[e])
    g = evaluate.submit(config_path, wait_for=[f])
    h = report.submit(config_path, wait_for=[g])
    return h.result()


if __name__ == "__main__":
    if "--serve" in sys.argv:
        recommendation_pipeline.serve(name="daily-recommendation-pipeline", cron="0 2 * * *")
    else:
        recommendation_pipeline()


In [ ]:
%%writefile recommendation-pipeline/scripts/__init__.py
# (package marker)


In [ ]:
%%writefile recommendation-pipeline/scripts/make_run_summary.py
"""Render the DAG of the last run (status + duration per task) from reports/run_log.jsonl -> reports/orchestration_run.png."""
import json
import subprocess
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
ORDER = ["ingest", "validate", "prepare", "transform", "features", "train", "evaluate", "report"]
COLORS = {"COMPLETED": "#c8e6c9", "FAILED": "#ffcdd2", "NOT RUN": "#eeeeee"}


def make():
    rows = {}
    for line in (ROOT / "reports" / "run_log.jsonl").read_text().splitlines():
        r = json.loads(line)
        rows[r["stage"]] = r
    total = sum(r["seconds"] for r in rows.values())
    ok = all(rows.get(s, {}).get("status") == "COMPLETED" for s in ORDER)
    dot = ['digraph G { rankdir=LR; bgcolor="white"; node [shape=box, style="rounded,filled", fontname="Helvetica", fontsize=11, margin="0.18,0.1"]; edge [color="#555555"];',
           f'labelloc=t; fontname="Helvetica-Bold"; fontsize=14; label="Flow run: recommendation-pipeline  |  {"Completed" if ok else "FAILED"}  |  total {total:.1f}s";']
    for s in ORDER:
        r = rows.get(s)
        st = r["status"] if r else "NOT RUN"
        dot.append(f'{s} [label="{s}\\n{st}\\n{r["seconds"]:.1f}s" , fillcolor="{COLORS[st]}"];' if r else
                   f'{s} [label="{s}\\nNOT RUN", fillcolor="{COLORS[st]}"];')
    dot.append(" -> ".join(ORDER) + ";")
    dot.append("}")
    src = ROOT / "reports" / "orchestration_run.dot"
    src.write_text("\n".join(dot))
    try:
        subprocess.run(["dot", "-Tpng", "-Gdpi=170", str(src), "-o", str(ROOT / "reports" / "orchestration_run.png")], check=True)
    except (FileNotFoundError, subprocess.CalledProcessError):      # no graphviz binary (e.g. Kaggle) -> matplotlib
        _matplotlib_fallback(rows, ok, total)
    src.unlink()


def _matplotlib_fallback(rows, ok, total):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyBboxPatch
    fig, ax = plt.subplots(figsize=(14, 2.2))
    ax.set_xlim(0, len(ORDER) * 1.5); ax.set_ylim(0, 1.6); ax.axis("off")
    ax.set_title(f"Flow run: recommendation-pipeline | {'Completed' if ok else 'FAILED'} | total {total:.1f}s", fontweight="bold")
    for i, s in enumerate(ORDER):
        r = rows.get(s)
        st = r["status"] if r else "NOT RUN"
        x = i * 1.5 + 0.1
        ax.add_patch(FancyBboxPatch((x, 0.35), 1.15, 0.8, boxstyle="round,pad=0.02,rounding_size=0.12", fc=COLORS[st], ec="#555"))
        ax.text(x + 0.575, 0.75, f"{s}\n{st}" + (f"\n{r['seconds']:.1f}s" if r else ""), ha="center", va="center", fontsize=9)
        if i < len(ORDER) - 1:
            ax.annotate("", xy=(x + 1.45, 0.75), xytext=(x + 1.17, 0.75), arrowprops=dict(arrowstyle="->", color="#555"))
    fig.savefig(ROOT / "reports" / "orchestration_run.png", dpi=150, bbox_inches="tight")
    plt.close(fig)


if __name__ == "__main__":
    make()


In [ ]:
%%writefile recommendation-pipeline/scripts/make_architecture.py
"""Generate reports/architecture.png (Graphviz) - run:  python scripts/make_architecture.py"""
import subprocess
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
DOT = r'''
digraph A {
  rankdir=TB; bgcolor="white"; nodesep=0.32; ranksep=0.55; newrank=true;
  node [shape=box, style="rounded,filled", fontname="Helvetica", fontsize=10, fillcolor="#e8f0fe", color="#5b7db1", margin="0.12,0.07"];
  edge [color="#555555", fontname="Helvetica", fontsize=8];
  labelloc=t; fontname="Helvetica-Bold"; fontsize=14; label="End-to-End Recommendation Data Pipeline  (all stages orchestrated by Prefect, cron 02:00 daily)";

  src  [label="Sources\nCSV / synthetic\nusers | items | interactions", shape=cylinder, fillcolor="#fff3cd", color="#b8962e"];
  ing  [label="1. Ingestion\nload as strings\nrow counts + MD5 log"];
  val  [label="2. Validation\nnulls, dtypes, duplicates,\nrating range, timestamps,\nreferential integrity, quality gate"];
  prep [label="3. Preparation\nimpute, standardise,\nde-duplicate,\ntime-based split"];
  tr   [label="4. Transformation\nuser activity, item popularity,\nuser-item sparse matrix,\nitem-genre matrix"];
  feat [label="5. Feature management\nregistry + offline store\nuser | item | pair features"];
  train[label="6. Training\npopularity | ALS MF | hybrid\n(params, metrics, artifacts\n-> MLflow)"];
  ev   [label="7. Evaluation\nP@K R@K NDCG@K\ncoverage, diversity,\nRMSE / MAE"];
  err  [label="data/errors/\nrejected rows + reasons\nreports/validation_report", shape=note, fillcolor="#fde2e2", color="#c05050"];
  fs   [label="data/features/\nParquet + feature catalog", shape=cylinder, fillcolor="#e6f4ea", color="#4a8f5c"];
  mod  [label="models/*.pkl", shape=cylinder, fillcolor="#e6f4ea", color="#4a8f5c"];
  ml   [label="MLflow (SQLite)\nruns, params, metrics", shape=component, fillcolor="#f3e8ff", color="#8a5bb5"];
  out  [label="reports/\nresults table + plots", shape=note, fillcolor="#e6f4ea", color="#4a8f5c"];
  srv  [label="recommend.py\nprediction CLI\n(same FeatureStore)", fillcolor="#fff3cd", color="#b8962e"];

  { rank=same; src -> ing -> val -> prep; }
  { rank=same; ev -> train -> feat -> tr [dir=back]; }
  { rank=same; err; ml; fs; }
  { rank=same; out; mod; srv; }
  prep -> tr;
  val -> err [style=dashed];
  feat -> fs [style=dashed]; fs -> train [style=dashed, label="reused"];
  train -> ml [style=dashed]; ev -> ml [style=dashed];
  train -> mod [style=dashed]; mod -> ev [style=dashed]; mod -> srv [style=dashed]; fs -> srv [style=dashed, label="reused"];
  ev -> out [style=dashed];
}
'''
(ROOT / "reports" / "architecture.dot").write_text(DOT)
subprocess.run(["dot", "-Tpng", "-Gdpi=170", str(ROOT / "reports" / "architecture.dot"), "-o", str(ROOT / "reports" / "architecture.png")], check=True)
(ROOT / "reports" / "architecture.dot").unlink()
print("wrote reports/architecture.png")


In [ ]:
%%writefile recommendation-pipeline/tests/test_pipeline.py
"""Unit tests for validation rules, metrics, and split logic.  Run: pytest -q"""
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from src.evaluation import metrics as m  # noqa: E402
from src.preparation.prepare import clean_interactions, time_split  # noqa: E402
from src.validation.validate import validate_interactions, validate_users  # noqa: E402

V = {"rating_min": 1, "rating_max": 5, "age_min": 10, "age_max": 100}
NOW = pd.Timestamp("2025-01-01")


def _users():
    raw = pd.DataFrame({"user_id": ["1", "2", "2", "x", "5"], "age": ["30", None, None, "40", "200"],
                        "gender": ["m", "f", "f", "m", "m"], "country": ["IN"] * 5,
                        "signup_date": ["2023-01-01"] * 5})
    return validate_users(raw, V, NOW)


def test_user_validation_flags_dup_dtype_and_keeps_warnings():
    clean, c = _users()
    assert sorted(clean.user_id) == [1, 2, 5]            # exact duplicate of '2' and non-integer 'x' rejected
    fails = {r["check"]: r["failed"] for r in c.results}
    assert fails["exact_duplicate_row"] == 1 and fails["invalid_dtype_user_id"] == 1
    assert fails["missing_age"] == 2 and fails["age_out_of_range"] == 1   # warnings: rows kept


def test_interaction_validation_rules():
    users, _ = _users()
    items = pd.DataFrame({"item_id": [10, 11]})
    raw = pd.DataFrame({
        "user_id": ["1", "1", "1", "1", "1", "1", "9"], "item_id": ["10", "10", "11", "11", "10", "99", "10"],
        "rating": ["4", "6", "five", None, "3", "3", "3"],
        "timestamp": ["2024-01-01 10:00:00", "2024-01-01 10:00:00", "2024-01-01 10:00:00", "2024-01-01 10:00:00",
                      "2031-01-01 00:00:00", "2024-01-01 10:00:00", "2024-01-01 10:00:00"]})
    clean, c = validate_interactions(raw, V, NOW, users, items)
    assert len(clean) == 1 and clean.iloc[0].rating == 4.0
    fails = {r["check"]: r["failed"] for r in c.results}
    assert fails["rating_out_of_range"] == 1 and fails["invalid_dtype_rating"] == 1 and fails["missing_rating"] == 1
    assert fails["future_timestamp"] == 1 and fails["orphan_item_id"] == 1 and fails["orphan_user_id"] == 1


def test_latest_rating_wins_and_split_is_chronological():
    x = pd.DataFrame({"user_id": [1, 1, 2, 3], "item_id": [5, 5, 6, 7], "rating": [2.0, 5.0, 3.0, 4.0],
                      "timestamp": pd.to_datetime(["2024-01-01", "2024-02-01", "2024-03-01", "2024-04-01"])})
    x = clean_interactions(x)
    assert len(x) == 3 and x[(x.user_id == 1)].rating.iloc[0] == 5.0
    tr, te, cut = time_split(x, 0.34)
    assert tr.timestamp.max() < te.timestamp.min()


def test_ranking_metrics_known_values():
    topk = np.array([[1, 2, 3]])
    r = m.ranking_metrics(topk, [{1, 3}], 3)
    assert abs(r["precision@3"] - 2 / 3) < 1e-9 and abs(r["recall@3"] - 1.0) < 1e-9
    expected = (1 + 1 / np.log2(4)) / (1 + 1 / np.log2(3))
    assert abs(r["ndcg@3"] - expected) < 1e-9


def test_topk_excludes_seen_and_coverage_diversity():
    scores = np.array([[0.9, 0.8, 0.7, 0.1]])
    tk = m.top_k(scores, np.array([[True, False, False, False]]), 2)
    assert list(tk[0]) == [1, 2]
    assert m.coverage(np.array([[0, 1], [1, 2]]), 4) == 0.75
    G = np.array([[1, 0], [1, 0], [0, 1]], float)
    assert m.diversity(np.array([[0, 1]]), G) == 0.0 and m.diversity(np.array([[0, 2]]), G) == 1.0


### Pipeline architecture

In [ ]:
import base64, os
PNG = "..."
open(os.path.join(PROJECT, "reports", "architecture.png"), "wb").write(base64.b64decode(PNG))

## 3. Dependencies

In [ ]:
import importlib.util, subprocess, sys
missing = [p for p in ["mlflow", "prefect", "pyarrow", "yaml"] if importlib.util.find_spec(p) is None]
if missing:
    pkgs = [{"yaml": "pyyaml"}.get(p, p) for p in missing]
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], capture_output=True, text=True)
    print("installed:", pkgs, "" if r.returncode == 0 else r.stderr[-800:])
else:
    print("all libraries available")

## 4. Unit tests

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], cwd=PROJECT, capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-500:])

## 5. Running the pipeline
The stages are executed as a Prefect flow (`dags/recommendation_flow.py`). If Prefect cannot start, the same stages are run sequentially.

In [ ]:
import os, re
env = {**os.environ, "PREFECT_SERVER_ANALYTICS_ENABLED": "false", "PYTHONUNBUFFERED": "1"}

def run(args):
    p = subprocess.run([sys.executable, "run_pipeline.py", *args], cwd=PROJECT, env=env, capture_output=True, text=True)
    return p.returncode, p.stdout + p.stderr

rc, out, mode = 1, "", "prefect not available"
if importlib.util.find_spec("prefect") is not None:
    rc, out = run([])
    mode = "Prefect flow"
if rc != 0:
    print(out[-1200:])
    rc, out = run(["--no-prefect"])
    mode = "sequential (no Prefect)"
lines = [re.sub(r"\x1b\[[0-9;]*m", "", l) for l in out.splitlines()]
dotted = re.compile(r"^\d\d:\d\d:\d\d\.\d{3} \|")
keep = [l for l in lines if "INFO mlflow" not in l and "tune w_" not in l
        and (not dotted.match(l) or "Finished in state" in l)]
print("Run mode:", mode, "| exit code:", rc)
print("\n".join(keep)[-7000:])
assert rc == 0

## 6. Results

### 6.1 Data validation report

In [ ]:
from IPython.display import Markdown, Image, display
rep = os.path.join(PROJECT, "reports")
display(Markdown(open(os.path.join(rep, "validation_report.md")).read()))

### 6.2 Model evaluation

In [ ]:
display(Markdown(open(os.path.join(rep, "evaluation_results.md")).read()))
display(Image(os.path.join(rep, "model_comparison.png")))

### 6.3 Feature catalog

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 80, "display.width", 200)
display(pd.read_csv(os.path.join(rep, "feature_catalog.csv")))

### 6.4 Architecture and orchestration run

In [ ]:
display(Image(os.path.join(rep, "architecture.png")))
display(Image(os.path.join(rep, "orchestration_run.png")))

### 6.5 MLflow tracking

In [ ]:
runs = pd.read_csv(os.path.join(rep, "mlflow_runs_summary.csv"))
cols = ["tags.mlflow.runName", "metrics.ndcg_at_10", "metrics.precision_at_10", "metrics.recall_at_10", "metrics.coverage_at_10",
        "metrics.diversity_at_10", "metrics.rmse", "metrics.mae", "metrics.train_seconds"]
display(runs[[c for c in cols if c in runs.columns]].round(4))

### 6.6 Example recommendations

In [ ]:
for m in ["hybrid", "popularity"]:
    r = subprocess.run([sys.executable, "recommend.py", "--user", "42", "--model", m, "--k", "5"], cwd=PROJECT, capture_output=True, text=True)
    print(m); print(r.stdout or r.stderr)

## 7. Output package

In [ ]:
import shutil
shutil.make_archive(os.path.join(BASE, "recommendation-pipeline-output"), "zip", BASE, "recommendation-pipeline")
print(os.path.getsize(os.path.join(BASE, "recommendation-pipeline-output.zip")) // 1000, "KB")